# Chapitre 7 - Cours 1 : erreur d'apprentissage, erreur de test et compromis biais-variance

Ce premier notebook du chapitre 7 (*Model Assessment and Selection*) pose le vocabulaire
de base :

1. les différentes **erreurs** d'un modèle (erreur d'apprentissage, erreur de test
   conditionnelle, erreur espérée) et la figure 7.1 du livre ;
2. le découpage apprentissage / validation / test ;
3. la **décomposition biais-variance** (éq. (7.9) à (7.14) du livre), démontrée pas à pas
   et vérifiée par simulation ;
4. la reproduction de la figure 7.3 : biais, variance et erreur de prédiction pour les
   $k$ plus proches voisins et pour la régression linéaire sur sous-ensembles, en
   régression **et** en classification.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import ch07_selection as sel
set_style()

## 1. Notations : fonction de perte et erreurs

On dispose d'un **échantillon d'apprentissage**
$\mathcal T = \{(x_1, y_1), \dots, (x_N, y_N)\}$ tiré d'une loi jointe $F$ du couple
$(X, Y)$, et d'une règle de prédiction $\hat f$ construite à partir de $\mathcal T$.
Une **fonction de perte** $L(Y, \hat f(X))$ mesure le coût d'une erreur. Les deux
choix usuels en régression (éq. (7.1) du livre) sont

$$
L(Y, \hat f(X)) = (Y - \hat f(X))^2 \quad\text{(perte quadratique)}, \qquad
L(Y, \hat f(X)) = |Y - \hat f(X)| \quad\text{(perte absolue)}.
$$

Le livre distingue **trois** quantités qu'il ne faut pas confondre.

* L'**erreur de test conditionnelle** (ou erreur de généralisation), éq. (7.2) :
  $$\mathrm{Err}_{\mathcal T} = \mathbb E\big[L(Y, \hat f(X)) \,\big|\, \mathcal T\big].$$
  L'échantillon $\mathcal T$ est **fixé** ; l'espérance porte sur un nouveau couple
  $(X, Y)$ indépendant de $\mathcal T$.
* L'**erreur espérée**, éq. (7.3) :
  $$\mathrm{Err} = \mathbb E\big[L(Y, \hat f(X))\big] = \mathbb E\big[\mathrm{Err}_{\mathcal T}\big],$$
  qui moyenne en plus sur tous les échantillons d'apprentissage possibles.
* L'**erreur d'apprentissage**, éq. (7.4) :
  $$\overline{\mathrm{err}} = \frac1N \sum_{i=1}^N L(y_i, \hat f(x_i)),$$
  calculée sur les données mêmes qui ont servi à construire $\hat f$.

> **Intuition.** $\mathrm{Err}_{\mathcal T}$ est ce qui nous intéresse vraiment (la
> qualité du modèle que *nous* avons ajusté), $\mathrm{Err}$ est plus facile à étudier
> mathématiquement, et $\overline{\mathrm{err}}$ est la seule que l'on puisse calculer
> directement… mais elle est **trop optimiste** : un modèle très flexible peut
> rendre $\overline{\mathrm{err}}$ nulle tout en prédisant mal de nouvelles données.